# 02.02 Taxonomy and Category Mapping

**Phase 02 — Data Preparation**  
*OFPHIL V2: Charity Patient Eligibility Classification*

---

### Description & Purpose
Standardize non-uniform categorical values, align medical diagnoses and charity eligibility classifications with standard taxonomies (MSWD / DSWD guidelines).


### Research Objectives

1. Standardize categorical labels (e.g., gender representations, employment types, housing ownership).
2. Map non-standard social work assessment categories to official MSWD/DSWD charity eligibility classifications.
3. Document explicit mapping dictionaries and taxonomy crosswalks.
4. Validate that no semantic loss occurs during category normalization.


### Inputs & Expected Outputs

**Required Inputs:**
- `Raw tabular records or interim validated tabular data.`
- `Institutional charity eligibility classification rubrics (e.g., Classes A, B, C1, C2, C3, D).`

**Expected Outputs:**
- `Taxonomy mapping dictionaries: `data/interim/cleaned/category_mappings.json`.`
- `Crosswalk verification report.`


### 1. Environment Setup & Library Imports

In [ ]:
from pathlib import Path
import json, datetime, re
import pandas as pd
import numpy as np

def _find_project_root(start: Path) -> Path:
    for p in [start, *start.parents]:
        if (p / 'data').exists() and (p / 'experiments').exists():
            return p
    return Path('../../..').resolve()

PROJECT_ROOT = _find_project_root(Path.cwd())

### 1. Setup Paths and Mapping Dictionaries

In [ ]:
SYNTH_CSV = PROJECT_ROOT / 'data' / 'synthetic' / 'tabular' / 'ofphil_synthetic_raw.csv'
CLEANED_DIR = PROJECT_ROOT / 'data' / 'interim' / 'cleaned'
CLEANED_DIR.mkdir(parents=True, exist_ok=True)
AUDIT_CSV_PATH = PROJECT_ROOT / 'data' / 'interim' / 'taxonomy_mapping' / 'category_mapping_log.csv'
AUDIT_CSV_PATH.parent.mkdir(parents=True, exist_ok=True)
df = pd.read_csv(SYNTH_CSV)

### 2. Define Standard Eligibility Classification Mapping

In [ ]:
eligibility_taxonomy = {'ELIGIBLE': 'ELIGIBLE', 'INELIGIBLE': 'INELIGIBLE'}
employment_mapping = {
    'Unemployed': 'Unemployed', 'Employed': 'Employed', 'Self-Employed': 'Self-Employed',
    'Retired': 'Retired', 'Pensioner': 'Pensioner', 'Student': 'Student',
}
residence_mapping = {
    'Owner': 'Owned', 'Owned': 'Owned', 'Renter': 'Rented',
    'Informal Settler': 'Informal Settler', 'Rent-free': 'Rent-Free',
}
marital_mapping = {
    'Married': 'Married', 'Single': 'Single', 'Widowed': 'Widowed',
    'Separated': 'Separated', 'Common-law partner': 'Common-Law',
    'Common-Law': 'Common-Law', 'Live-in': 'Common-Law',
}

### 3. Standardize Socioeconomic Categorical Features

In [ ]:
mapping_defs = [
    ('eligibility', eligibility_taxonomy, 'target_standardization'),
    ('employment_status', employment_mapping, 'employment_taxonomy'),
    ('residence_status', residence_mapping, 'housing_standardization'),
    ('marital_status', marital_mapping, 'civil_status_normalization'),
]
log_rows = []
for col, cmap, rule in mapping_defs:
    vc = df[col].dropna().value_counts()
    for raw_val, cnt in vc.items():
        std_val = cmap.get(str(raw_val), str(raw_val))
        log_rows.append({
            'column_name': col,
            'raw_value': str(raw_val),
            'standardized_value': std_val,
            'record_count': int(cnt),
            'mapping_rule': rule,
        })
pd.DataFrame(log_rows).to_csv(AUDIT_CSV_PATH, index=False)
print(f'Saved {len(log_rows)} category mapping entries -> {AUDIT_CSV_PATH}')

### 4. Export Category Mapping Artifacts

In [ ]:
mapping_artifact = CLEANED_DIR / 'category_mappings.json'
mapping_artifact.write_text(json.dumps({
    'eligibility': eligibility_taxonomy,
    'employment': employment_mapping,
    'residence': residence_mapping,
    'marital_status': marital_mapping,
}, indent=2), encoding='utf-8')
print(f'Category mappings exported to: {mapping_artifact}')

### Findings & Observations

- **Observations:** Document mapping ambiguities, obsolete codes encountered in historical data, and alignment decisions made with institutional social work guidelines.
- **Data Integrity & Governance Notes:** Verified that no raw patient identifiers or unmasked confidential records are displayed.


### Completion Checklist

- [ ] Eligibility categories mapped to official institutional definitions.
- [ ] Socioeconomic categoricals harmonized across years.
- [ ] No unmapped orphan categories remaining.
- [ ] Mapping json artifact preserved for downstream inference.
